In [21]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.ticker as plticker
import seaborn as sns
import pandas as pd
from pandas.io.parsers import read_csv
import numpy as np
#from anndata import AnnData, read_h5ad
#import singlecellmultiomics.bamProcessing.bamToRNACounts
#import loompy
import scanpy as sc
import scipy
import math
#import pyreadr
from scipy import stats

In [3]:
%config Completer.use_jedi = False


In [4]:
# scripts for fig1c

In [7]:
#load files

chicreads=pd.read_csv('input/CellReadCount_merged_tagged.bam.csv', header=0, index_col=0)

transreads1=pd.read_csv('input/CellReadCount_PZ-MB-TChIC-K562-H3K4me3-CT.csv', header=0, index_col=0)
transreads2=pd.read_csv('input/CellReadCount_PZ-TChIC-K562-SC-H3K27me3-CT.csv', header=0, index_col=0)

# Combine them
transreads1 = pd.concat([transreads1, transreads2])



H3K4me3_QC=pd.read_csv('input/QCselector_H3K4me3_round1.csv', header=0, index_col=0, sep='\t')
H3K27me3_QC=pd.read_csv('input/QCselector_H3K27me3_round1.csv', header=0, index_col=0, sep='\t')

In [8]:
#combine chic and trans

transreads1.columns=('RNA_reads','RNA_reads2')
chicreads.columns=('ChIC_reads','ChIC_reads2')
trans=transreads1['RNA_reads']
chic=chicreads['ChIC_reads']
countspercell=transreads1.join(chicreads, how='inner')
countspercell["RNA_log10"]=np.log10(countspercell['RNA_reads2']+1)
countspercell["ChIC_log10"]=np.log10(countspercell['ChIC_reads']+1)


In [9]:
#integrate QC output
goodH3K4me3=H3K4me3_QC[H3K4me3_QC=="good"].index
goodH3K27me3=H3K27me3_QC[H3K27me3_QC=="good"].index
countspercell['QC']='bad'
countspercell.loc[goodH3K4me3,'QC']='good'
countspercell.loc[goodH3K27me3,'QC']='good'
countspercell.loc[countspercell['RNA_log10']<3,'QC']='bad'

In [10]:
#transfer QC state into sample names
countspercell['sample']='unkonwn'
matching = [s for s in countspercell.index if "H3K4me3-CT_" in s]
countspercell.loc[matching,'sample']='H3K4me3 + RNA'
matching = [s for s in countspercell.index if "H3K27me3-CT_" in s]
countspercell.loc[matching,'sample']='H3K27me3 + RNA'
countspercell_main=countspercell

countspercell_main.loc[((countspercell_main['sample']=='H3K4me3 + RNA')&(countspercell_main['QC']=='good')),'sample_name']='H3K4me3 used'
countspercell_main.loc[((countspercell_main['sample']=='H3K4me3 + RNA')&(countspercell_main['QC']=='bad')),'sample_name']='H3K4me3 filtered'
countspercell_main.loc[((countspercell_main['sample']=='H3K27me3 + RNA')&(countspercell_main['QC']=='good')),'sample_name']='H3K27me3 used'
countspercell_main.loc[((countspercell_main['sample']=='H3K27me3 + RNA')&(countspercell_main['QC']=='bad')),'sample_name']='H3K27me3 filtered'

In [11]:
Green='#238b45'
Orange='#d94801'
Purple='#6a51a3'

In [12]:
#plot version fig1c
sns.set_style("whitegrid", {'axes.grid' : False, 'xtick.bottom': True,
 'ytick.left': True, 'axes.edgecolor': 'black'})
sns.set_context("talk", font_scale=1.1)
plt.figure(figsize=(6,6))

ax=sns.scatterplot(data=countspercell_main, x='RNA_log10', y='ChIC_log10', hue='sample_name', palette=[Green,'#79c793',Purple,'#9D77F7'],size=20)

ax.grid(False)
ax.legend([], frameon=False)
ax.set(xlabel ="RNA reads [log10]", ylabel = "ChIC reads [log10]")

plt.savefig('TCHIC_fig1_d.pdf', dpi=300, format='pdf', bbox_inches='tight')

In [13]:
countspercell_main.groupby('sample_name')['ChIC_reads'].median()

sample_name
H3K27me3 filtered       44.0
H3K27me3 used        27246.0
H3K4me3 filtered       529.0
H3K4me3 used          3825.0
Name: ChIC_reads, dtype: float64

In [14]:
countspercell_main.groupby('sample_name')['RNA_reads'].median()

sample_name
H3K27me3 filtered       89.0
H3K27me3 used         9545.0
H3K4me3 filtered     11602.0
H3K4me3 used         14411.5
Name: RNA_reads, dtype: float64

In [15]:
#script for fig1d, figs1b, figs1c

In [22]:
#load RNA loom files

H3K4me3_trans = sc.read_loom('input/PZ-MB-TChIC-K562-H3K4me3-CT.loom',obs_names='CellID')
H3K27me3_trans = sc.read_loom('input/PZ-TChIC-K562-SC-H3K27me3-CT.loom',obs_names='CellID')




In [26]:
#change cellindex to the same format as the ChIC count tables

split=H3K4me3_trans.obs.index.str.split(":",expand=True,)
split=split.to_frame()
split=split.reset_index()
plate=split[0]
plate.index=H3K4me3_trans.obs.index
H3K4me3_trans.obs["batch"]=plate

barcodes_df = pd.read_csv("input/VASAbarcodes.csv", sep='\t', header=None, index_col=0)
barcodes = barcodes_df.iloc[:, 0].to_dict()
barcodes = {y:x for x,y in barcodes.items()}
H3K4me3_trans.obs['cellnames'] = ['-'.join(x) for x in [ob.split('-')[0:7] for ob in H3K4me3_trans.obs.index]]
H3K4me3_trans.obs['bc'] = [ob.split(':')[1] for ob in H3K4me3_trans.obs['cellnames']]
H3K4me3_trans.obs['bc'] = H3K4me3_trans.obs['bc'].map(lambda x: barcodes[x])
H3K4me3_trans.obs['cellname'] = H3K4me3_trans.obs['batch'].astype(str) + '_' + H3K4me3_trans.obs['bc'].astype(str)
H3K4me3_trans.obs.index = H3K4me3_trans.obs['cellname']
H3K4me3_trans.obs.index.rename('index')

H3K4me3_trans.obs.index = H3K4me3_trans.obs.index.rename('index')


split=H3K27me3_trans.obs.index.str.split(":",expand=True,)
split=split.to_frame()
split=split.reset_index()
plate=split[0]
plate.index=H3K27me3_trans.obs.index
H3K27me3_trans.obs["batch"]=plate


H3K27me3_trans.obs['cellnames'] = ['-'.join(x) for x in [ob.split('-')[0:7] for ob in H3K27me3_trans.obs.index]]
H3K27me3_trans.obs['bc'] = [ob.split(':')[1] for ob in H3K27me3_trans.obs['cellnames']]
H3K27me3_trans.obs['bc'] = H3K27me3_trans.obs['bc'].map(lambda x: barcodes[x])
H3K27me3_trans.obs['cellname'] = H3K27me3_trans.obs['batch'].astype(str) + '_' + H3K27me3_trans.obs['bc'].astype(str)
H3K27me3_trans.obs.index = H3K27me3_trans.obs['cellname']
H3K27me3_trans.obs.index.rename('index')

H3K27me3_trans.obs.index = H3K27me3_trans.obs.index.rename('index')

In [27]:
# remove duplicate genes
H3K27me3_trans=H3K27me3_trans[:,~H3K27me3_trans.var.index.duplicated(keep='first')]
H3K4me3_trans=H3K4me3_trans[:,~H3K4me3_trans.var.index.duplicated(keep='first')]

In [28]:
#load ChIC TSS count tables and add to adata files

H3K4me3_P=pd.read_csv('input/PZ-MB-TChIC-K562-H3K4me3-CT-chic.bam.TSS_10kb.csv', header=0, index_col=(0,1,2,3))
H3K4me3_P = H3K4me3_P.iloc[1:]
H3K4me3_P.index.names = ['reference_name','start','end', 'bname']
H3K4me3_PW = H3K4me3_P.droplevel(['start', 'end', 'reference_name'], axis=0)
H3K4me3_PW = H3K4me3_PW[~H3K4me3_PW.index.duplicated(keep='first')]
k4MatrixForAdata = H3K4me3_PW.T.merge(H3K4me3_trans.obs['cellname'], 
                left_index=True,
                right_index=True, 
                how="right").fillna(0).drop('cellname', 
                                            axis = 1).T.merge(H3K4me3_trans.var['Accession'],
                                                              left_index=True,right_index=True,
                                                             how="right").fillna(0).drop('Accession', axis=1).reindex(H3K4me3_trans.var.index).T
H3K4me3_trans.layers['H3K4me3'] = scipy.sparse.csr_matrix(k4MatrixForAdata,dtype=np.float32)

H3K27me3_P=pd.read_csv('input/PZ-MB-TChIC-K562-H3K27me3-CT-chic.bam.TSS_10kb.csv', header=0, index_col=(0,1,2,3))
H3K27me3_P = H3K27me3_P.iloc[1:]
H3K27me3_P.index.names = ['reference_name','start','end', 'bname']
H3K27me3_PW = H3K27me3_P.droplevel(['start', 'end', 'reference_name'], axis=0)
H3K27me3_PW = H3K27me3_PW[~H3K27me3_PW.index.duplicated(keep='first')]
k27MatrixForAdata = H3K27me3_PW.T.merge(H3K27me3_trans.obs['cellnames'], 
                left_index=True,
                right_index=True, 
                how="right").fillna(0).drop('cellnames', 
                                            axis = 1).T.merge(H3K27me3_trans.var['Accession'],
                                                              left_index=True,right_index=True,
                                                             how="right").fillna(0).drop('Accession', axis=1).reindex(H3K27me3_trans.var.index).T
H3K27me3_trans.layers['H3K27me3'] = scipy.sparse.csr_matrix(k27MatrixForAdata,dtype=np.float32)

In [30]:
#import gene annotation file to select protein coding genes

genes=pd.read_csv('input/Homo_sapiens.GRCh38.93.gene.bed', header=None, index_col=None, sep='\t')
genes.index=genes[6]
genelengths=abs(genes[1]-genes[2])
protgenelist=genes[genes[8]=='protein_coding'].index
protgenelist=protgenelist[~protgenelist.duplicated(keep='first')]



In [31]:
#select only protein coding genes to avoid impacts from multimappers
H3K27me3_trans=H3K27me3_trans[:,~H3K27me3_trans.var.index.duplicated(keep='first')]
H3K27me3_trans=H3K27me3_trans[:,protgenelist]
matching2=[s for s in H3K27me3_trans.var.index if "MT-" in s]
H3K27me3_trans=H3K27me3_trans[:,~H3K27me3_trans.var.index.isin(matching2)]
H3K27me3_trans

H3K4me3_trans=H3K4me3_trans[:,~H3K4me3_trans.var.index.duplicated(keep='first')]
H3K4me3_trans=H3K4me3_trans[:,protgenelist]
matching2=[s for s in H3K4me3_trans.var.index if "MT-" in s]
H3K4me3_trans=H3K4me3_trans[:,~H3K4me3_trans.var.index.isin(matching2)]
H3K4me3_trans

View of AnnData object with n_obs × n_vars = 379 × 19957
    obs: 'batch', 'cellnames', 'bc', 'cellname'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand'
    layers: 'matrix', 'ambiguous', 'spliced', 'unspliced', 'H3K4me3'

In [32]:
#QC select cells using QC file from part 1

H3K27me3_trans=H3K27me3_trans[countspercell_main[countspercell_main['sample_name']=='H3K27me3 used'].index]
H3K4me3_trans=H3K4me3_trans[countspercell_main[countspercell_main['sample_name']=='H3K4me3 used'].index]


In [33]:
# combine transcript data in one file to determine gene order by pseudobulk transcription strength
RNAsums=pd.DataFrame(H3K27me3_trans.layers['matrix'].toarray())
RNAsums.columns=H3K27me3_trans.var.index.copy()
RNAsumsK27=RNAsums.sum(axis=0)
RNAsums2=pd.DataFrame(H3K4me3_trans.layers['matrix'].toarray())
RNAsums2.columns=H3K4me3_trans.var.index.copy()
RNAsumsK4=RNAsums2.sum(axis=0)
RNAsumsfinal=RNAsumsK27+RNAsumsK4
RNAsumsfinallog=np.log1p(RNAsumsfinal)

In [36]:
# pseudobulk counts per cell and transcript quantile
RNA27table=pd.DataFrame()
K27table=pd.DataFrame()
genetable=pd.DataFrame()
for i in range(0,9):
    genelist=RNAsumsfinallog[((RNAsumsfinallog<=((RNAsumsfinallog.max()/10)*(10-i)))&(RNAsumsfinallog>((RNAsumsfinallog.max()/10)*(9-i))))].index
    dataset=H3K27me3_trans[:,H3K27me3_trans.var.index.isin(genelist)].copy()
    transdataset=pd.DataFrame(columns=['values','mark','quantile'])
    transdataset['values']=pd.DataFrame(dataset.layers['matrix'].toarray()).mean(axis=1)
    transdataset.index=dataset.obs.index
    transdataset['mark']='RNA'
    transdataset['quantile']=i
    RNA27table=pd.concat([RNA27table, transdataset])
    
    chromdataset=pd.DataFrame(columns=['values','mark','quantile'])
    chromdataset['values']=pd.DataFrame(dataset.layers['H3K27me3'].toarray()).mean(axis=1)
    chromdataset.index=dataset.obs.index
    chromdataset['mark']='H3K27me3'
    chromdataset['quantile']=i
    K27table=pd.concat([K27table, chromdataset])

genelist=RNAsumsfinallog[RNAsumsfinallog<=(RNAsumsfinallog.max()/10)].index
dataset=H3K27me3_trans[:,H3K27me3_trans.var.index.isin(genelist)].copy()
transdataset=pd.DataFrame(columns=['values','mark','quantile'])
transdataset['values']=pd.DataFrame(dataset.layers['matrix'].toarray()).mean(axis=1)
transdataset.index=dataset.obs.index
transdataset['mark']='RNA'
transdataset['quantile']=9
RNA27table=pd.concat([RNA27table, transdataset])
    
chromdataset=pd.DataFrame(columns=['values','mark','quantile'])
chromdataset['values']=pd.DataFrame(dataset.layers['H3K27me3'].toarray()).mean(axis=1)
chromdataset.index=dataset.obs.index
chromdataset['mark']='H3K27me3'
chromdataset['quantile']=9
K27table=pd.concat([K27table, chromdataset])
        

RNA4table=pd.DataFrame()
K4table=pd.DataFrame()
genetable=pd.DataFrame()
for i in range(0,9):
    genelist=RNAsumsfinallog[((RNAsumsfinallog<=((RNAsumsfinallog.max()/10)*(10-i)))&(RNAsumsfinallog>((RNAsumsfinallog.max()/10)*(9-i))))].index
    dataset=H3K4me3_trans[:,H3K4me3_trans.var.index.isin(genelist)].copy()
    transdataset=pd.DataFrame(columns=['values','mark','quantile'])
    transdataset['values']=pd.DataFrame(dataset.layers['matrix'].toarray()).mean(axis=1)
    transdataset.index=dataset.obs.index
    transdataset['mark']='RNA'
    transdataset['quantile']=i
    RNA4table=pd.concat([RNA4table, transdataset])
    
    chromdataset=pd.DataFrame(columns=['values','mark','quantile'])
    chromdataset['values']=pd.DataFrame(dataset.layers['H3K4me3'].toarray()).mean(axis=1)
    chromdataset.index=dataset.obs.index
    chromdataset['mark']='H3K4me3'
    chromdataset['quantile']=i
    K4table=pd.concat([K4table, chromdataset])

genelist=RNAsumsfinallog[RNAsumsfinallog<=(RNAsumsfinallog.max()/10)].index
dataset=H3K4me3_trans[:,H3K4me3_trans.var.index.isin(genelist)].copy()
transdataset=pd.DataFrame(columns=['values','mark','quantile'])
transdataset['values']=pd.DataFrame(dataset.layers['matrix'].toarray()).mean(axis=1)
transdataset.index=dataset.obs.index
transdataset['mark']='RNA'
transdataset['quantile']=9
RNA4table=pd.concat([RNA4table, transdataset])
    
chromdataset=pd.DataFrame(columns=['values','mark','quantile'])
chromdataset['values']=pd.DataFrame(dataset.layers['H3K4me3'].toarray()).mean(axis=1)
chromdataset.index=dataset.obs.index
chromdataset['mark']='H3K4me3'
chromdataset['quantile']=9
K4table=pd.concat([K4table, chromdataset])
        

In [37]:
#scale data so it can be plotted on same y axis and save in new column

RNA27table['values_scaled']=RNA27table['values']/np.nanquantile(RNA27table['values'],0.98)
RNA27table['values_scaled_cut']=RNA27table['values_scaled']
RNA27table.loc[RNA27table['values_scaled_cut']>1,'values_scaled_cut']=1
K27table['values_scaled']=K27table['values']/np.nanquantile(K27table['values'],0.98)
K27table['values_scaled_cut']=K27table['values_scaled']
K27table.loc[K27table['values_scaled_cut']>1,'values_scaled_cut']=1


RNA4table['values_scaled']=RNA4table['values']/np.nanquantile(RNA4table['values'],0.98)
RNA4table['values_scaled_cut']=RNA4table['values_scaled']
RNA4table.loc[RNA4table['values_scaled_cut']>1,'values_scaled_cut']=1
K4table['values_scaled']=K4table['values']/np.nanquantile(K4table['values'],0.98)
K4table['values_scaled_cut']=K4table['values_scaled']
K4table.loc[K4table['values_scaled_cut']>1,'values_scaled_cut']=1

In [39]:
#merge tables for plotting

plottable = pd.concat([RNA27table, K27table, RNA4table, K4table], ignore_index=True)

In [40]:
sns.set_style("whitegrid", {'axes.grid' : False, 'xtick.bottom': True,
 'ytick.left': True, 'axes.edgecolor': 'black'})
sns.set_context("talk", font_scale=1.1)
plt.figure(figsize=(8,6))

#ax=sns.stripplot(y="values_scaled", 
#                x="quantile", 
#                data=plottable, hue='mark', hue_order=['RNA','H3K4me3','H3K27me3'], dodge=True,
#              palette=[Orange,Green,Purple], alpha=0.1, jitter=0.2)

ax=sns.boxplot(y="values_scaled_cut", 
                x="quantile", hue='mark', hue_order=['RNA','H3K4me3','H3K27me3'],
                data=plottable, palette=[Orange,Green,Purple], fliersize=0)



ax.legend([], frameon=False)
ax.invert_xaxis()


#sns.move_legend(
#    ax, "lower center",
#    bbox_to_anchor=(.5, 1), ncol=3, title=None, frameon=False,
#)
ax.set_ylim([-0.05,1.05])
plt.savefig('TCHIC_fig1_c.pdf', dpi=300, format='pdf', bbox_inches='tight')

In [41]:
sns.set_context("talk", font_scale=1)
sns.set_style("whitegrid", {'axes.grid' : False, 'xtick.bottom': True,
 'ytick.left': True, 'axes.edgecolor': 'black'})

fig, axes = plt.subplots(4, 1, figsize=(8, 15), gridspec_kw={'hspace': 0.1})
#plt.figure(figsize=(40,200))
axes = axes.flatten()
#fig.tight_layout(pad=0)

ax = sns.boxplot(data=plottable[plottable['mark']=='RNA'], x='quantile', y='values',orient='v',
    ax=axes[0], width=0.9, fliersize=0,color=Orange)
#ax=sns.stripplot(y="values", 
#                x="quantile", ax=axes[0],
#                data=plottable[plottable['mark']=='RNA'], dodge=True,
#              palette=['Orange'], alpha=0.1, jitter=0.2)
#loc = plticker.MultipleLocator(base=0.5) # this locator puts ticks at regular intervals
#ax.yaxis.set_major_locator(loc)
ax.tick_params(axis='x', rotation=90)
ax.tick_params(labelbottom=False, bottom=False)
axes[0].set_yscale('log')
ax.set_ylim([0.00009,100])
ax.invert_xaxis()
ax.set(xlabel='',
       ylabel='RNA mean counts',
       title='')


ax = sns.boxplot(data=plottable[plottable['mark']=='RNA'], x='quantile', y='values', orient='v', 
    ax=axes[1], width=0.9,fliersize=0 , color=Orange)
#ax=sns.stripplot(y="values", 
#                x="quantile", ax=axes[1],
#                data=plottable[plottable['mark']=='RNA'], dodge=True,
#              palette=['Orange'], alpha=0.1, jitter=0.2)
ax.set_ylim([-1,30])
ax.tick_params(axis='x', rotation=90)
ax.tick_params(labelbottom=False, bottom=False)
ax.set(xlabel='',
       ylabel='RNA mean counts',
       title='')
ax.invert_xaxis()


ax = sns.boxplot(data=plottable[plottable['mark']=='H3K4me3'], x='quantile', y='values', orient='v', 
    ax=axes[2], width=0.9,fliersize=0 ,color=Green)
#ax=sns.stripplot(y="values", 
#                x="quantile", ax=axes[2],
#                data=plottable[plottable['mark']=='H3K4me3'], dodge=True,
#              palette=['Green'], alpha=0.1, jitter=0.2)
ax.set_ylim([-0.02,0.6])
ax.tick_params(axis='x', rotation=90)
ax.tick_params(labelbottom=False, bottom=False)
#ax.tick_params(labelleft=False)
ax.set(xlabel='',
       ylabel='H3K4me3 mean counts',
       title='')
ax.invert_xaxis()


ax = sns.boxplot(data=plottable[plottable['mark']=='H3K27me3'], x='quantile', y='values',orient='v',
    ax=axes[3], width=0.9, fliersize=0,color=Purple)
#ax=sns.stripplot(y="values", 
#                x="quantile", ax=axes[3],
#                data=plottable[plottable['mark']=='H3K27me3'], dodge=True,
#              palette=['Purple'], alpha=0.1, jitter=0.2)
ax.set_ylim([-0.02,0.6])
ax.tick_params(axis='x')


ax.set(xlabel='',
       ylabel='H3K27me3 mean counts',
       title='')
ax.invert_xaxis()


plt.savefig('TCHIC_figS1_b.pdf', dpi=300, format='pdf', bbox_inches='tight')


In [42]:
#method comparison figs1 d and figs e

In [43]:
#load and split datatables 
mcomp1=pd.read_csv('input/methodcomp1.csv', header=0, index_col=None, sep=',')
mcomp1k4=mcomp1[mcomp1.target=='H3K4me3']
mcomp1k27=mcomp1[mcomp1.target=='H3K27me3']


In [44]:
markers = {"ours_2023": "o", "rang_2022": "s", "zhu_2021": "X", "xiong_2021": "P"}
sns.set_context("talk", font_scale=1.1)
palette1 = sns.color_palette("Purples",4)
palette2 = ['grey','cornflowerblue','gold','mediumaquamarine']
plt.figure(figsize=(4,5))
ax=sns.boxplot(y="total_counts_chromatin", 
                x="study", palette=palette2,
                data=mcomp1k27, fliersize=0, order=['ours_2023', 'rang_2022','zhu_2021','xiong_2021'])
plt.axhline(y = (mcomp1k27.loc[mcomp1k27['study']=="ours_2023","total_counts_chromatin"]).median(), 
            color = 'grey', linestyle = 'dashed')


#ax.set_ylim([0,6])
ax.set(xlabel='',
       ylabel='H3K27me3 counts',
       title='')
plt.xticks(rotation=90)
plt.yscale('log')
ax.set_ylim([1,1000000])
ax.tick_params(labelbottom=False, bottom=False)

plt.savefig('TCHIC_figS1_d4.pdf', dpi=300, format='pdf', bbox_inches='tight')

In [45]:
mcomp1k27.groupby(['study'])['total_counts_chromatin'].median()

study
ours_2023     81570.5
rang_2022      8332.0
xiong_2021      644.5
zhu_2021        606.0
Name: total_counts_chromatin, dtype: float64

In [46]:
markers = {"ours_2023": "o", "rang_2022": "s", "zhu_2021": "X", "xiong_2021": "P"}
sns.set_context("talk", font_scale=1.1)
palette1 = sns.color_palette("Purples",4)
palette2 = ['grey','gold','mediumaquamarine']
plt.figure(figsize=(4,5))
ax=sns.boxplot(y="total_counts_chromatin", 
                x="study", palette=palette2,
                data=mcomp1k4, fliersize=0, order=['ours_2023', 'zhu_2021','xiong_2021'])
plt.axhline(y = (mcomp1k4.loc[mcomp1k4['study']=="ours_2023","total_counts_chromatin"]).median(), 
            color = 'grey', linestyle = 'dashed')


plt.yscale('log')
ax.set_ylim([1,100000])
ax.set(xlabel='',
       ylabel='H3K4me3 counts',
       title='')
plt.xticks(rotation=90)
ax.tick_params(labelbottom=False, bottom=False)

plt.savefig('TCHIC_figS1_d3.pdf', dpi=300, format='pdf', bbox_inches='tight')

In [47]:
mcomp1k4.groupby(['study'])['total_counts_chromatin'].median()

study
ours_2023     4049.0
xiong_2021     910.0
zhu_2021       447.0
Name: total_counts_chromatin, dtype: float64

In [48]:
markers = {"ours_2023": "o", "rang_2022": "s", "zhu_2021": "X", "xiong_2021": "P"}
sns.set_context("talk", font_scale=1.1)
palette1 = sns.color_palette("Purples",4)
palette2 = ['grey','cornflowerblue','gold','mediumaquamarine']
plt.figure(figsize=(4,5))
ax=sns.boxplot(y="total_counts_rna", 
                x="study", palette=palette2,
                data=mcomp1, fliersize=0, order=['ours_2023', 'rang_2022', 'zhu_2021','xiong_2021'])
plt.axhline(y = (mcomp1.loc[mcomp1['study']=="ours_2023","total_counts_rna"]).median(), 
            color = 'grey', linestyle = 'dashed')


plt.yscale('log')
ax.set_ylim([1,100000])
ax.set(xlabel='',
       ylabel='RNA counts',
       title='')
plt.xticks(rotation=90)
ax.tick_params(labelbottom=False, bottom=False)

plt.savefig('TCHIC_figS1_d1.pdf', dpi=300, format='pdf', bbox_inches='tight')

In [49]:
mcomp1.groupby(['study'])['total_counts_rna'].median()

study
ours_2023     6389.0
rang_2022     5257.5
xiong_2021    2146.0
zhu_2021       813.0
Name: total_counts_rna, dtype: float64

In [50]:
markers = {"ours_2023": "o", "rang_2022": "s", "zhu_2021": "X", "xiong_2021": "P"}
sns.set_context("talk", font_scale=1.1)
palette1 = sns.color_palette("Purples",4)
palette2 = ['grey','cornflowerblue','gold','mediumaquamarine']
plt.figure(figsize=(4,5))
ax=sns.boxplot(y="ngenes_nonzero", 
                x="study", palette=palette2,
                data=mcomp1, fliersize=0, order=['ours_2023', 'rang_2022', 'zhu_2021','xiong_2021'])
plt.axhline(y = (mcomp1.loc[mcomp1['study']=="ours_2023","ngenes_nonzero"]).median(), 
            color = 'grey', linestyle = 'dashed')


#plt.yscale('log')
ax.set_ylim([-1000,10000])
ax.set(xlabel='',
       ylabel='RNA genes',
       title='')
plt.xticks(rotation=90)
ax.tick_params(labelbottom=False, bottom=False)

plt.savefig('TCHIC_figS1_d2.pdf', dpi=300, format='pdf', bbox_inches='tight')

In [51]:
mcomp1.groupby(['study'])['ngenes_nonzero'].median()

study
ours_2023     3109.0
rang_2022     2371.5
xiong_2021     877.0
zhu_2021       554.0
Name: ngenes_nonzero, dtype: float64

In [53]:
#load data for figs e

mcompRNA=pd.read_csv('input/RNA_mean_quantile.csv', header=0, index_col=None, sep=',')
for i in mcompRNA['Study'].unique():
    mcompRNA.loc[mcompRNA['Study']==i,'value_scaled']=mcompRNA.loc[mcompRNA['Study']==i,'value']/np.nanquantile(mcompRNA.loc[mcompRNA['Study']==i,'value'],0.98)
mcompRNA['Mark']='RNA'


mcompchrom=pd.read_csv('input/chromatin_mean_quantile.csv', header=0, index_col=None, sep=',')

mcompchromK4=mcompchrom[mcompchrom['Mark']=='H3K4me3']
for i in mcompchromK4['Study'].unique():
    mcompchromK4.loc[mcompchromK4['Study']==i,'value_scaled']=mcompchromK4.loc[mcompchromK4['Study']==i,'value']/np.nanquantile(mcompchromK4.loc[mcompchromK4['Study']==i,'value'],0.98)

mcompchromK27=mcompchrom[mcompchrom['Mark']=='H3K27me3']
for i in mcompchromK27['Study'].unique():
    mcompchromK27.loc[mcompchromK27['Study']==i,'value_scaled']=mcompchromK27.loc[mcompchromK27['Study']==i,'value']/np.nanquantile(mcompchromK27.loc[mcompchromK27['Study']==i,'value'],0.98)

mcompchrom=pd.concat([mcompchromK4, mcompchromK27])
mcompall=pd.concat([mcompRNA, mcompchrom])


mcompall['value_scaled_cut']=mcompall['value_scaled']
mcompall.loc[mcompall['value_scaled_cut']>1,'value_scaled_cut']=1

In [54]:
#plot version used

sns.set_context("talk", font_scale=1)
sns.set_style("whitegrid", {'axes.grid' : False, 'xtick.bottom': True,
 'ytick.left': True, 'axes.edgecolor': 'black'})

fig, axes = plt.subplots(2, 2, figsize=(20, 10), gridspec_kw={'wspace': 0.02, 'hspace': 0.2})
#plt.figure(figsize=(40,200))
axes = axes.flatten()
#fig.tight_layout(pad=0)

ax = sns.boxplot(data=mcompall[mcompall['Study']=='ours_2023'], x='quantile', y='value_scaled_cut', hue='Mark', hue_order=['RNA', 'H3K4me3','H3K27me3'],orient='v',
    ax=axes[0], width=0.9, fliersize=0,palette=[Orange,Green,Purple])
#ax = sns.stripplot(y="value_scaled", 
#                x="quantile", ax=axes[0],hue='Mark',
#                data=mcompall[mcompall['Study']=='ours_2023'], dodge=True,
#              palette=['Orange','Green','Purple'], alpha=0.05, jitter=0.2)
ax.set_ylim([-0.05,1.05])
ax.tick_params(axis='x', rotation=90)

ax.invert_xaxis()
ax.tick_params(labelbottom=False, bottom=False)

ax.set(xlabel='',
       ylabel='scaled mean counts',
       title='')
sns.move_legend(
    ax, "lower center",
    bbox_to_anchor=(-0.1, 1), ncol=3, title=None, frameon=False,
)
ax.get_legend().remove()                                                                    

sns.set_style("whitegrid", {'axes.grid' : False, 'xtick.bottom': True,
 'ytick.left': False})



ax = sns.boxplot(data=mcompall[mcompall['Study']=='rang_2022'], x='quantile', y='value_scaled_cut', hue='Mark', hue_order=['RNA', 'H3K4me3','H3K27me3'],orient='v',
    ax=axes[1], width=0.9, fliersize=0,palette=[Orange,Green,Purple])
#ax = sns.stripplot(y="value_scaled", 
#                x="quantile", ax=axes[1],hue='Mark',
#                data=mcompall[mcompall['Study']=='rang_2022'], hue_order=['RNA', 'H3K4me3','H3K27me3'], dodge=True,
#              palette=['Orange','Green','Purple'], alpha=0.05, jitter=0.2)
ax.set_ylim([-0.05,1.05])
ax.tick_params(axis='x', rotation=90)
ax.tick_params(labelleft=False, left=False)
ax.tick_params(labelbottom=False, bottom=False)
ax.set(xlabel='',
       ylabel='',
       title='')
ax.get_legend().remove()
ax.invert_xaxis()





ax = sns.boxplot(data=mcompall[mcompall['Study']=='zhu_2021'], x='quantile', y='value_scaled_cut', hue='Mark', hue_order=['RNA', 'H3K4me3','H3K27me3'],orient='v',
    ax=axes[2], width=0.9, fliersize=0,palette=[Orange,Green,Purple])
#ax = sns.stripplot(y="value_scaled", 
#                x="quantile", ax=axes[2],hue='Mark',
#                data=mcompall[mcompall['Study']=='zhu_2021'], dodge=True,
#              palette=['Orange','Green','Purple'], alpha=0.05, jitter=0.2)
ax.set_ylim([-0.05,1.05])
ax.tick_params(axis='x', rotation=90)
ax.set(xlabel='',
       ylabel='scaled mean counts',
       title='')
ax.get_legend().remove()
ax.invert_xaxis()

ax = sns.boxplot(data=mcompall[mcompall['Study']=='xiong_2021'], x='quantile', y='value_scaled_cut', hue='Mark', hue_order=['RNA', 'H3K4me3','H3K27me3'],orient='v',
    ax=axes[3], width=0.9, fliersize=0,palette=[Orange,Green,Purple])
#ax = sns.stripplot(y="value_scaled", 
#                x="quantile", ax=axes[3],hue='Mark',
#                data=mcompall[mcompall['Study']=='xiong_2021'], dodge=True,
#              palette=['Orange','Green','Purple'], alpha=0.05, jitter=0.2)
ax.set_ylim([-0.05,1.05])
ax.tick_params(axis='x', rotation=90)

ax.tick_params(labelleft=False, left=False)
ax.set(xlabel='',
       ylabel='',
       title='')
ax.get_legend().remove()
ax.invert_xaxis()


plt.savefig('TCHIC_figS1_e.pdf', dpi=300, format='pdf', bbox_inches='tight')

In [55]:
# scripts for figs1b load data

chic_all=pd.read_csv('input/full_chic.csv', header=0, index_col=[0,1,2])
trans_all=pd.read_csv('input/full_trans.csv', header=0, index_col=[0,1,2])


In [56]:
#select H3K4me3 data and combine for plotting and correlation coefficinient calculation

matching = [s for s in chic_all.columns if "TChIC-K562-H3K4me3" in s]
TCHIC_K4=chic_all[matching]
TCHIC_K4=TCHIC_K4.sum(axis=1).to_frame()
matching = [s for s in chic_all.columns if "sortChIC-k562-k4me3" in s]
sortCHIC_K4=chic_all[matching]
sortCHIC_K4=sortCHIC_K4.sum(axis=1).to_frame()
TCHIC_K4.columns={'TChIC'}
TCHIC_K4=(TCHIC_K4/TCHIC_K4.sum())*1000000
sortCHIC_K4.columns={'sortChIC'}
sortCHIC_K4=(sortCHIC_K4/sortCHIC_K4.sum())*1000000
K4_scatterdata=TCHIC_K4.join(sortCHIC_K4, how='outer')
res = stats.pearsonr(K4_scatterdata['TChIC'], K4_scatterdata['sortChIC'])
res

PearsonRResult(statistic=np.float64(0.9061776602223836), pvalue=np.float64(0.0))

In [57]:
res = stats.spearmanr(K4_scatterdata['TChIC'], K4_scatterdata['sortChIC'])
res

SignificanceResult(statistic=np.float64(0.7122584752657177), pvalue=np.float64(0.0))

In [58]:
sns.set_style("whitegrid", {'axes.grid' : False, 'xtick.bottom': True,
 'ytick.left': True, 'axes.edgecolor': 'black'})
sns.set_context("talk", font_scale=1.1)
plt.figure(figsize=(6,6))

ax=sns.scatterplot(data=K4_scatterdata, x='sortChIC', y='TChIC', alpha=0.2, color=Green, size=10)
plt.xscale('log')
plt.yscale('log')
ax.legend([],frameon=False)

plt.savefig('TCHIC_figS1_c2.pdf', dpi=300, format='pdf', bbox_inches='tight')

In [59]:
#select H3K27me3 data and combine for plotting and correlation coefficinient calculation


matching = [s for s in chic_all.columns if "TChIC-K562-SC-H3K27me3" in s]
TCHIC_K27=chic_all[matching]
TCHIC_K27=TCHIC_K27.sum(axis=1).to_frame()
matching = [s for s in chic_all.columns if "sortChIC-k562-k27me3" in s]
sortCHIC_K27=chic_all[matching]
sortCHIC_K27=sortCHIC_K27.sum(axis=1).to_frame()
TCHIC_K27.columns={'TChIC'}
TCHIC_K27=(TCHIC_K27/TCHIC_K27.sum())*1000000
sortCHIC_K27.columns={'sortChIC'}
sortCHIC_K27=(sortCHIC_K27/sortCHIC_K27.sum())*1000000
K27_scatterdata=TCHIC_K27.join(sortCHIC_K27, how='outer')
#K27_scatterdata=K27_scatterdata.fillna(0)
res = stats.pearsonr(K27_scatterdata['TChIC'], K27_scatterdata['sortChIC'])
res

PearsonRResult(statistic=np.float64(0.8964306390409698), pvalue=np.float64(0.0))

In [60]:
res = stats.spearmanr(K27_scatterdata['TChIC'], K27_scatterdata['sortChIC'])
res

SignificanceResult(statistic=np.float64(0.9502545476610084), pvalue=np.float64(0.0))

In [61]:
plt.figure(figsize=(6,6))
ax=sns.scatterplot(data=K27_scatterdata, x='sortChIC', y='TChIC', alpha=0.2, color=Purple,size=10)
plt.xscale('log')
plt.yscale('log')
#ax.set_ylim([-10,150])
#ax.set_xlim([-10,150])
ax.legend([],frameon=False)

plt.savefig('TCHIC_figS1_c3.pdf', dpi=300, format='pdf', bbox_inches='tight')

In [62]:
#select transcript data and combine for plotting and correlation coefficinient calculation


matching = [s for s in trans_all.columns if "TChIC-K562-H3K4me3" in s]
TCHIC_K4_T=trans_all[matching]
TCHIC_K4_T=TCHIC_K4_T.sum(axis=1).to_frame()
matching = [s for s in trans_all.columns if "JVL-VASA1" in s]
VASA=trans_all[matching]
VASA=VASA.sum(axis=1).to_frame()
TCHIC_K4_T.columns={'TChIC'}
TCHIC_K4_T=(TCHIC_K4_T/TCHIC_K4_T.sum())*1000000

VASA.columns={'VASA'}
VASA=(VASA/VASA.sum())*1000000
K4_T_scatterdata=TCHIC_K4_T.join(VASA, how='outer')
K4_T_scatterdata['mark']='H3K4me3'
res = stats.pearsonr(K4_T_scatterdata['TChIC'], K4_T_scatterdata['VASA'])
res

PearsonRResult(statistic=np.float64(0.6453342709341917), pvalue=np.float64(0.0))

In [63]:
res = stats.spearmanr(K4_T_scatterdata['TChIC'], K4_T_scatterdata['VASA'])
res

SignificanceResult(statistic=np.float64(0.8759749929818542), pvalue=np.float64(0.0))

In [64]:
plt.figure(figsize=(6,6))
ax=sns.scatterplot(data=K4_T_scatterdata, x='VASA', y='TChIC',alpha=0.2, color=Orange,size=10)

plt.xscale('log')
plt.yscale('log')
ax.legend([],frameon=False)

plt.savefig('TCHIC_figS1_c1.pdf', dpi=300, format='pdf', bbox_inches='tight')